# Phenogram plot

> **Under development** — not published in docs. Requires GWASLab from source (`sys.path` below).

Create a **phenogram** (karyotype-style ideogram) showing cytogenetic bands for each autosome with genome-wide lead variants marked and optionally annotated.

**Prerequisites**
- Sumstats must include `CHR`, `POS`, and `P` (or `MLOG10P`)
- Genomic build must match cytobands: `build="19"` (hg19) or `build="38"` (hg38)
- Only autosomes chr1–chr22 are plotted
- Layout: chr1–chr11 form the **top** row (left to right); chr12–chr22 the row below. Ideogram pter caps are **top-aligned** within each row. Annotations in the right column are laid out as **rigid units** (marker cluster + label, or a wrapped text block): each unit moves as one piece on a bounded 1D vertical stack solver—markers and labels are not spread separately.
- `anno="GENENAME"` auto-downloads Ensembl GTF on first use

## Load GWASLab

In [1]:
import sys
sys.path.insert(0, "/home/yunye/work/gwaslab/src")
import gwaslab as gl

In [2]:
gl.show_version()

2026/06/11 18:10:39 GWASLab v4.1.9 https://cloufield.github.io/gwaslab/
2026/06/11 18:10:39 (C) 2022-2026, Yunye He, Kamatani Lab, GPL-3.0 license, gwaslab@gmail.com
2026/06/11 18:10:39 Python version: 3.12.0 | packaged by conda-forge | (main, Oct  3 2023, 08:43:22) [GCC 12.3.0]


## Load sample sumstats

In [ ]:
mysumstats = gl.Sumstats(
    "../../0_sample_data/t2d_bbj.txt.gz",
    snpid="SNP",
    chrom="CHR",
    pos="POS",
    ea="ALT",
    nea="REF",
    neaf="Frq",
    beta="BETA",
    se="SE",
    p="P",
    direction="Dir",
    build="19",
    n="N",
)
mysumstats.basic_check(verbose=False)

2026/06/11 18:10:40 GWASLab v4.1.9 https://cloufield.github.io/gwaslab/
2026/06/11 18:10:40 (C) 2022-2026, Yunye He, Kamatani Lab, GPL-3.0 license, gwaslab@gmail.com
2026/06/11 18:10:40 Python version: 3.12.0 | packaged by conda-forge | (main, Oct  3 2023, 08:43:22) [GCC 12.3.0]
2026/06/11 18:10:40  -Top 3 inferred source formats:
2026/06/11 18:10:40    1. pheweb [DETECTED] (score: 23.608) | 2. gwaslab (score: 20.789) | 3. ldsc (score: 18.116)
2026/06/11 18:10:40 Start to initialize gl.Sumstats from file :../../0_sample_data/t2d_bbj.txt.gz
2026/06/11 18:10:52  -Reading columns          : N,BETA,SE,SNP,CHR,P,Frq,POS,Dir,REF,ALT
2026/06/11 18:10:52  -Renaming columns to      : N,BETA,SE,SNPID,CHR,P,EAF,POS,DIRECTION,NEA,EA
2026/06/11 18:10:52  -Current Dataframe shape : 12557761  x  11
2026/06/11 18:10:52  -Initiating a status column: STATUS ...
2026/06/11 18:10:52  -Genomic coordinates are based on GRCh37/hg19...
2026/06/11 18:10:54  -NEAF is specified...
2026/06/11 18:10:54  -Checking 

## Basic phenogram (lead bars only)

By default, `plot_phenogram()` extracts lead variants via `_get_sig()` (500 kb windows, P < 5e-8) and draws red bars on the ideogram without text labels.

In [ ]:
mysumstats.plot_phenogram(repel_force=2)

## Text labels with gene names

Set `anno="GENENAME"` to label leads with the nearest protein-coding gene. Use `repel_force` to control vertical spacing between labels.

In [ ]:
mysumstats.plot_phenogram(anno="GENENAME", repel_force=0.01, ncols=11)

## Text labels with SNPID subset

Use `anno_set` to restrict which leads receive **text labels** by **SNPID** (not gene names). All extracted leads still get red bars.

First inspect lead SNPIDs, then pass exact strings to `anno_set`:

In [ ]:
lead_snps = mysumstats.get_lead(windowsizekb=500, sig_level=5e-8)
lead_snps[["SNPID", "CHR", "POS", "P"]].head(10)

In [ ]:
anno_set = lead_snps.sort_values("P").head(3)["SNPID"].tolist()
print("anno_set:", anno_set)

mysumstats.plot_phenogram(
    anno="SNPID",
    anno_set=anno_set,
    anno_max_rows=10,
    repel_force=1.5,
    ncols=12,
)

## Marker mode (grouped loci)

When `anno_group` is set, phenogram switches to **marker mode**: variants are grouped by a column, with shape and color mapped from additional columns. Set `use_lead_extraction=False` when the input table already lists the variants to plot.

Below we build a synthetic multi-locus table with:

- **Dense clusters** on chr1, chr11, and chr19 (separate loci only ~150–800 kb apart on the same chromosome)
- **Wrap groups** with 5–8 markers per locus (more than four per row)
- Multi-hit loci, partial overlap, solo loci, and duplicate locus names on different chromosomes

In [ ]:
import numpy as np
import pandas as pd

CHR_SIZES = {
    1: 249_250_621,  2: 243_199_373,  3: 198_022_430,  4: 191_154_276,
    5: 180_915_260,  6: 171_115_067,  7: 159_138_663,  8: 146_364_022,
    9: 141_213_431, 10: 135_534_747, 11: 135_006_516, 12: 133_851_895,
    13: 115_169_878, 14: 107_349_540, 15: 102_531_392, 16: 90_354_753,
    17: 81_195_210, 18: 78_077_248, 19: 59_128_983, 20: 63_025_520,
    21: 48_129_895, 22: 51_304_566,
}
rng = np.random.default_rng(42)
subtypes = ["TypeA", "TypeB", "TypeC"]
ancestries = ["EUR", "EAS", "AFR", "SAS"]

def add_row(rows, rs_i, chrom, pos, locus, subtype, ancestry, p_exp):
    rows.append({
        "SNPID": f"rs{rs_i}",
        "CHR": chrom,
        "POS": int(pos),
        "P": 10 ** -p_exp,
        "LOCUS": locus,
        "SUBTYPE": subtype,
        "ANCESTRY": ancestry,
    })
    return rs_i + 1

def add_group(rows, rs_i, chrom, base, locus, n_variants, pos_jitter=30_000, p_range=(12, 40)):
    """Add ``n_variants`` rows for one locus near ``base``."""
    base = min(int(base), CHR_SIZES[chrom] - max(pos_jitter + 10_000, 500_000))
    for _ in range(n_variants):
        pos = base + rng.integers(0, pos_jitter)
        rs_i = add_row(
            rows, rs_i, chrom, pos, locus,
            rng.choice(subtypes), rng.choice(ancestries),
            rng.uniform(*p_range),
        )
    return rs_i

rows = []
rs_i = 1

# Multi-hit loci (12 variants each: 3 subtypes x 4 ancestries)
overlap_loci = [
    ("FTO",     16, 53_800_000),
    ("APOE",    19, 45_410_000),
    ("TCF7L2",  10, 114_700_000),
    ("1p36",     1,  2_500_000),
]
for locus, chrom, base in overlap_loci:
    base = min(base, CHR_SIZES[chrom] - 500_000)
    for subtype in subtypes:
        for ancestry in ancestries:
            pos = base + rng.integers(0, 30_000)
            rs_i = add_row(rows, rs_i, chrom, pos, locus, subtype, ancestry,
                            rng.uniform(12, 40))

# Loci with 5–8 markers (exercises 4-per-row wrapping)
wrap_loci = [
    ("Wrap_01", 6, 31_000_000, 6),
    ("Wrap_02", 9, 20_000_000, 7),
    ("Wrap_03", 14, 55_000_000, 8),
]
for locus, chrom, base, n in wrap_loci:
    rs_i = add_group(rows, rs_i, chrom, base, locus, n, pos_jitter=20_000)

# Closely spaced loci on chr11 (~400–750 kb apart)
dense_chr11 = [
    ("Dense_A", 11, 58_000_000),
    ("Dense_B", 11, 58_550_000),
    ("Dense_C", 11, 59_100_000),
    ("Dense_D", 11, 59_650_000),
    ("Dense_E", 11, 60_200_000),
    ("Dense_F", 11, 60_750_000),
]
for locus, chrom, base in dense_chr11:
    rs_i = add_group(rows, rs_i, chrom, base, locus, 5, pos_jitter=12_000, p_range=(10, 22))

# Tight cluster on chr19 (~150–250 kb apart)
tight_chr19 = [
    ("Tight_1", 19, 45_100_000),
    ("Tight_2", 19, 45_280_000),
    ("Tight_3", 19, 45_460_000),
    ("Tight_4", 19, 45_640_000),
]
for locus, chrom, base in tight_chr19:
    rs_i = add_group(rows, rs_i, chrom, base, locus, 4, pos_jitter=8_000, p_range=(11, 28))

# chr1 cluster: four loci within ~3 Mb
dense_chr1 = [
    ("Chr1_A", 1, 40_000_000),
    ("Chr1_B", 1, 40_900_000),
    ("Chr1_C", 1, 41_800_000),
    ("Chr1_D", 1, 42_600_000),
]
for locus, chrom, base in dense_chr1:
    rs_i = add_group(rows, rs_i, chrom, base, locus, 6, pos_jitter=15_000)

partial_loci = [
    ("Locus_P1", 6, 30_000_000),
    ("Locus_P2", 11, 60_000_000),
]
for locus, chrom, base in partial_loci:
    base = min(base, CHR_SIZES[chrom] - 1_000_000)
    for subtype in subtypes[:2]:
        for ancestry in ancestries[:2]:
            pos = base + rng.integers(0, 500_000)
            rs_i = add_row(rows, rs_i, chrom, pos, locus, subtype, ancestry,
                            rng.uniform(10, 25))

solo_loci = [
    ("Solo_01",  2,  40_000_000),
    ("Solo_02",  4,  80_000_000),
    ("Solo_03",  7, 100_000_000),
    ("Solo_04", 12,  70_000_000),
    ("Solo_05", 17,  20_000_000),
    ("Solo_06", 20,  30_000_000),
    ("Solo_07", 21,  10_000_000),
    ("Solo_08", 22,  25_000_000),
]
for locus, chrom, pos in solo_loci:
    pos = min(pos, CHR_SIZES[chrom] - 500_000)
    rs_i = add_row(rows, rs_i, chrom, pos, locus,
                   rng.choice(subtypes), rng.choice(ancestries),
                   rng.uniform(8, 20))

for locus, chrom, pos in [("SharedName", 3, 50_000_000), ("SharedName", 8, 90_000_000)]:
    pos = min(pos, CHR_SIZES[chrom] - 500_000)
    for ancestry in ancestries[:2]:
        rs_i = add_row(rows, rs_i, chrom, pos + rng.integers(0, 10_000),
                       locus, "TypeA", ancestry, rng.uniform(10, 18))

leads = pd.DataFrame(rows)
ss = gl.Sumstats(leads, snpid="SNPID", chrom="CHR", pos="POS", p="P", build="19", verbose=False)
print(len(leads), "rows,", leads["LOCUS"].nunique(), "loci")
display(leads.groupby("LOCUS").agg(
    n=("SNPID", "count"),
    chr=("CHR", "first"),
    pos_min=("POS", "min"),
    pos_max=("POS", "max"),
    span_kb=("POS", lambda x: (x.max() - x.min()) // 1000),
).sort_values(["chr", "pos_min"]))

In [ ]:
ss.plot_phenogram(
    build="19",
    use_lead_extraction=False,
    anno_group="LOCUS",
    anno_shape="SUBTYPE",
    anno_color="ANCESTRY",
    ncols=11,
)

## Marker mode — adjust layout and styling

Marker mode exposes several layout knobs. The table below lists the most useful ones; all are passed to `plot_phenogram()`.

| Parameter | Default | Role |
|-----------|---------|------|
| `figsize` | (20, 48) | Figure width × height in inches; taller figures give stacked rows more vertical room (chr1 row on top) |
| `marker_size` | 81 | Scatter marker area |
| `marker_max_per_row` | 4 | Markers per row before wrapping |
| `marker_gap_pt` | 3 | Horizontal edge-to-edge gap within a row (pt) |
| `marker_row_gap_pt` | 2.5 | Vertical gap between wrapped marker rows (pt) |
| `marker_label_gap_pt` | 2.75 | Gap from markers to label (pt); also default spacing between adjacent annotation units |
| `marker_fontsize` | 11 | Group label font size |
| `marker_linewidth` | 0.6 | Marker edge width |
| `repel_force` | 0.5 | Scales vertical spreading for nearby loci |
| `group_min_vertical_gap_pt` | 3.5 | Minimum gap between group blocks (pt) |
| `group_marker_to_marker_gap_pt` | 3 | Extra spacing between adjacent groups (pt) |
| `anno_wrap_chars_per_line` | 9 | Wrap long group labels |
| `legend_kwargs` | — | e.g. `{"fontsize": 13, "markersize": 12}` |
| `anno_kwargs` | — | Matplotlib text styling (`fontstyle`, `fontweight`, …) |

The next cell uses a **taller figure**, **smaller markers**, **6 per row**, and tuned spreading so dense chr11/chr19 clusters remain readable.

In [ ]:
ss.plot_phenogram(
    build="19",
    use_lead_extraction=False,
    anno_group="LOCUS",
    anno="LOCUS",
    anno_shape="SUBTYPE",
    anno_color="ANCESTRY",
    ncols=8,
    figsize=(22, 56),
    # --- marker grid ---
    marker_size=64,
    marker_max_per_row=6,
    marker_gap_pt=2.0,
    marker_row_gap_pt=3.0,
    marker_label_gap_pt=3.5,
    marker_linewidth=0.8,
    # --- group labels ---
    marker_fontsize=10,
    anno_wrap_chars_per_line=12,
    anno_kwargs={"fontstyle": "normal", "fontweight": "bold"},
    # --- spreading for closely spaced loci ---
    repel_force=0.5,
    group_min_vertical_gap_pt=5.0,
    group_marker_to_marker_gap_pt=4.0,
    group_label_box_pad_pt=2.0,
    # --- legend ---
    legend_kwargs={"fontsize": 13, "markersize": 12},
)

## Save figure

Pass `save=True` (default filename) or a path string. Additional options go in `save_kwargs`.

In [ ]:
mysumstats.plot_phenogram(
    repel_force=2,
    save="phenogram_t2d.png",
    save_kwargs={"dpi": 300},
)